In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.linalg import hilbert
import warnings
warnings.filterwarnings('ignore')
%matplotlib inline

plt.rcParams.update({
    'figure.figsize': (8, 5),
    'axes.grid': True,
    'grid.alpha': 0.3,
    'font.size': 12
})

np.random.seed(42)
print('NumPy version:', np.__version__)
print('Setup complete.')

**The Matrix Inverse**

A natural first guess: invert each element individually. This fails because matrix multiplication is not element-wise.

In [ ]:
# Demonstrate: element-wise inversion is NOT the matrix inverse
A = np.array([[1, 4],
              [2, 7]])

A_elemwise = 1 / A  # element-wise reciprocal
A_true_inv = np.linalg.inv(A)

print(f'A =\n{A}')
print(f'\n1/A (element-wise) =\n{np.round(A_elemwise, 3)}')
print(f'\nA @ (1/A) =\n{np.round(A @ A_elemwise, 3)}')
print(f'NOT the identity!\n')

print(f'A^(-1) (true inverse) =\n{A_true_inv}')
print(f'\nA @ A^(-1) =\n{np.round(A @ A_true_inv, 10)}')
print(f'Identity! Confirmed.')

**Inverse of a 2 x 2 Matrix**

In [ ]:
# 2x2 inverse: manual formula vs NumPy
A = np.array([[1, 4],
              [2, 7]])

a, b, c, d = A[0,0], A[0,1], A[1,0], A[1,1]
det_A = a*d - b*c

# Manual formula
A_inv_manual = (1/det_A) * np.array([[d, -b], [-c, a]])
A_inv_numpy = np.linalg.inv(A)

print(f'A = {A.tolist()}')
print(f'det(A) = {a}*{d} - {b}*{c} = {det_A}')
print(f'\nA^(-1) (manual formula):\n{A_inv_manual}')
print(f'A^(-1) (NumPy):\n{A_inv_numpy}')
print(f'Match: {np.allclose(A_inv_manual, A_inv_numpy)}')

# Verify
print(f'\nA @ A^(-1) =\n{A @ A_inv_manual}')

In [ ]:
# Singular matrix: det = 0
B = np.array([[1, 4],
              [2, 8]])

det_B = B[0,0]*B[1,1] - B[0,1]*B[1,0]
print(f'B = {B.tolist()}')
print(f'det(B) = 1*8 - 4*2 = {det_B}')
print(f'rank(B) = {np.linalg.matrix_rank(B)}  (singular!)')

# Try to invert -> error
try:
    np.linalg.inv(B)
except np.linalg.LinAlgError as e:
    print(f'\nnp.linalg.inv error: {e}')

# But the pseudoinverse always exists
B_pinv = np.linalg.pinv(B)
print(f'\nPseudoinverse B^+ =\n{np.round(B_pinv, 6)}')
print(f'\nB @ B^+ =\n{np.round(B @ B_pinv, 4)}')
print(f'(Close to identity? No -- it is the projection matrix.)')

** Inverse of a Diagonal Matrix**

In [ ]:
# Diagonal inverse
D = np.diag([2, 3, 4])
D_inv = np.diag(1 / np.diag(D))
D_inv_numpy = np.linalg.inv(D)

print(f'D =\n{D}')
print(f'\nD^(-1) (invert each diagonal) =\n{np.round(D_inv, 4)}')
print(f'\nD @ D^(-1) =\n{D @ D_inv}')
print(f'Match numpy: {np.allclose(D_inv, D_inv_numpy)}')

# Diagonal with a zero: singular
D_singular = np.diag([2, 0, 4])
print(f'\nDiagonal with zero: rank = {np.linalg.matrix_rank(D_singular)}')
try:
    np.linalg.inv(D_singular)
except np.linalg.LinAlgError as e:
    print(f'Cannot invert: {e}')

When any diagonal element is zero, the matrix is singular (rank < N) and the inverse does not exist. In the pseudoinverse, we simply skip the zero singular values instead of inverting them -- this is the key insight behind the Moore-Penrose pseudoinverse.

**The MCA Algorithm: Inverting Any Square Full-Rank Matrix**

In [ ]:
def inverse_mca(A):
    """Compute inverse via Minors-Cofactors-Adjugate algorithm."""
    n = A.shape[0]
    assert A.shape[0] == A.shape[1], 'Matrix must be square'

    det_A = np.linalg.det(A)
    assert abs(det_A) > 1e-10, f'Matrix is singular (det={det_A:.2e})'

    # 1. Minors matrix
    minors = np.zeros((n, n))
    for i in range(n):
        for j in range(n):
            # Delete row i and column j
            sub = np.delete(np.delete(A, i, axis=0), j, axis=1)
            minors[i, j] = np.linalg.det(sub)

    # 2. Grid matrix
    grid = np.array([[(-1)**(i+j) for j in range(n)] for i in range(n)])

    # 3. Cofactors = Hadamard(minors, grid)
    cofactors = minors * grid

    # 4. Adjugate = cofactors^T / det(A)
    A_inv = cofactors.T / det_A
    return A_inv, minors, grid, cofactors

# Test on a 3x3 matrix
np.random.seed(42)
A = np.random.randint(-5, 6, size=(3, 3)).astype(float)
A_inv_mca, minors, grid, cofactors = inverse_mca(A)
A_inv_numpy = np.linalg.inv(A)

print(f'A =\n{A.astype(int)}')
print(f'det(A) = {np.linalg.det(A):.1f}')
print(f'\nMinors:\n{np.round(minors, 1)}')
print(f'\nGrid:\n{grid}')
print(f'\nCofactors (Minors * Grid):\n{np.round(cofactors, 1)}')
print(f'\nA^(-1) (MCA):\n{np.round(A_inv_mca, 6)}')
print(f'A^(-1) (NumPy):\n{np.round(A_inv_numpy, 6)}')
print(f'\nMatch: {np.allclose(A_inv_mca, A_inv_numpy)}')
print(f'A @ A^(-1) =\n{np.round(A @ A_inv_mca, 10)}')


When to use what:

Understanding: MCA builds intuition for how the inverse depends on determinants and cofactors.
Practice: Always use np.linalg.inv() or np.linalg.solve() (even better -- avoids explicit inversion).

**One-Sided Inverses**

In [ ]:
# Left-inverse of a tall matrix
np.random.seed(42)
T = np.random.randint(-10, 11, size=(40, 4)).astype(float)

print(f'T: {T.shape} (tall), rank = {np.linalg.matrix_rank(T)}')

# Step 1: T^T @ T is square and invertible
TtT = T.T @ T
print(f'T^T @ T: {TtT.shape}, rank = {np.linalg.matrix_rank(TtT)}')

# Step 2: Compute left-inverse
TtT_inv = np.linalg.inv(TtT)
L = TtT_inv @ T.T

print(f'L (left-inverse): {L.shape}')
print(f'\nL @ T (should be I_4):\n{np.round(L @ T, 10)}')

# T @ L is NOT the identity
TL = T @ L
print(f'\nT @ L shape: {TL.shape}')
print(f'T @ L == I_40? {np.allclose(TL, np.eye(40))}')

In [ ]:
# Right-inverse of a wide matrix
np.random.seed(42)
W = np.random.randint(-10, 11, size=(3, 8)).astype(float)

print(f'W: {W.shape} (wide), rank = {np.linalg.matrix_rank(W)}')

# Compute right-inverse
WWt = W @ W.T
WWt_inv = np.linalg.inv(WWt)
R = W.T @ WWt_inv

print(f'R (right-inverse): {R.shape}')
print(f'\nW @ R (should be I_3):\n{np.round(W @ R, 10)}')
print(f'\nR @ W == I_8? {np.allclose(R @ W, np.eye(8))}')

**The Moore-Penrose Pseudoinverse**

In [ ]:
# Pseudoinverse for square singular, tall, and wide matrices
matrices = {
    'Square singular (2x2)': np.array([[1, 4], [2, 8]], dtype=float),
    'Square full-rank (2x2)': np.array([[1, 4], [2, 7]], dtype=float),
    'Tall full col-rank (5x3)': np.random.randn(5, 3),
    'Wide full row-rank (3x5)': np.random.randn(3, 5),
}

for name, A in matrices.items():
    r = np.linalg.matrix_rank(A)
    A_pinv = np.linalg.pinv(A)
    prod = A @ A_pinv

    # How close to identity?
    target_I = np.eye(A.shape[0])
    err = np.linalg.norm(prod - target_I, 'fro')

    print(f'{name}: shape={A.shape}, rank={r}')
    print(f'  A @ A^+: {prod.shape}, ||A@A^+ - I||_F = {err:.6f}')

    # Check if pinv equals inv for full-rank square
    if A.shape[0] == A.shape[1] and r == A.shape[0]:
        A_inv = np.linalg.inv(A)
        print(f'  pinv == inv? {np.allclose(A_pinv, A_inv)}')
    print()

**Numerical Stability of the Inverse**

In [ ]:
# Hilbert matrices: condition number grows explosively
sizes = [3, 5, 8, 10, 13, 15]

print(f'{"N":>4s} {"cond(H)":>15s} {"rank":>5s} {"||H@inv(H) - I||":>18s}')
print('-' * 50)

for n in sizes:
    H = hilbert(n)
    cond = np.linalg.cond(H)
    r = np.linalg.matrix_rank(H)

    try:
        H_inv = np.linalg.inv(H)
        err = np.linalg.norm(H @ H_inv - np.eye(n), 'fro')
    except:
        err = float('inf')

    print(f'{n:>4d} {cond:>15.2e} {r:>5d} {err:>18.6e}')

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))

n = 5
H = hilbert(n)
H_inv = np.linalg.inv(H)
prod = H @ H_inv

for ax, M, title, cmap in zip(axes,
    [H, H_inv, prod],
    [f'Hilbert matrix $\mathbf{{H}}_{{5}}$\n(values: 0.04 to 1.0)',
     f'$\mathbf{{H}}_{{5}}^{{-1}}$\n(values: {H_inv.min():.0f} to {H_inv.max():.0f})',
     f'$\mathbf{{H}}_{{5}} \mathbf{{H}}_{{5}}^{{-1}}$\n(should be $\mathbf{{I}}_5$)'],
    ['YlOrBr', 'RdBu_r', 'gray_r']):

    im = ax.imshow(M, cmap=cmap, aspect='auto')
    plt.colorbar(im, ax=ax, shrink=0.8)
    ax.set_title(title, fontsize=11)
    ax.set_xticks(range(n)); ax.set_yticks(range(n))

plt.tight_layout()
plt.savefig('fig_hilbert.png', dpi=100, bbox_inches='tight')
plt.show()

print(f'H_5 element range: [{H.min():.4f}, {H.max():.4f}]')
print(f'H_5^(-1) element range: [{H_inv.min():.0f}, {H_inv.max():.0f}]')
print(f'Condition number: {np.linalg.cond(H):.2e}')

**Geometric Interpretation: The Inverse Undoes a Transformation**

In [ ]:
# Geometric demonstration: transform -> inverse-transform
# Original shape: a house
house = np.array([
    [0, 1, 1, 0.5, 0, 0],  # x
    [0, 0, 1, 1.5, 1, 0],  # y
])

T = np.array([[1.5, 0.5],
              [0.3, 1.2]])
T_inv = np.linalg.inv(T)

transformed = T @ house
recovered = T_inv @ transformed

fig, axes = plt.subplots(1, 3, figsize=(15, 5))

for ax, pts, title, color in zip(axes,
    [house, transformed, recovered],
    ['Original $\\mathbf{P}$',
     'Transformed $\\mathbf{Q} = \\mathbf{T}\\mathbf{P}$',
     'Recovered $\\mathbf{T}^{-1}\\mathbf{Q} = \\mathbf{P}$'],
    ['k', '#2196F3', '#4CAF50']):
    ax.fill(pts[0], pts[1], alpha=0.2, color=color)
    ax.plot(pts[0], pts[1], '-o', color=color, lw=2, ms=6)
    ax.set_title(title, fontsize=12)
    ax.set_xlim(-0.5, 3.5); ax.set_ylim(-0.5, 3)
    ax.set_aspect('equal')
    ax.axhline(0, color='k', lw=0.3); ax.axvline(0, color='k', lw=0.3)

plt.tight_layout()
plt.savefig('fig_inverse_geom.png', dpi=100, bbox_inches='tight')
plt.show()

print(f'T =\n{T}')
print(f'det(T) = {np.linalg.det(T):.2f}')
print(f'\nRecovered == Original? {np.allclose(recovered, house)}')

Cross-chapter connection: This geometric interpretation is the foundation of eigendecomposition (Chapter 13): diagonalising a matrix means finding the transformation that turns the matrix into a simple scaling (diagonal) matrix, which is easy to invert.

The inverse is unique (proof by contradiction), but the pseudoinverse of a singular matrix is not unique -- the Moore-Penrose pseudoinverse is the standard choice because it minimises the solution norm.